# 01 – Datenexploration

**Datensatz:** BPI Challenge 2017, Kreditanträge eines niederländischen Finanzinstituts (Januar 2016 bis Februar 2017).

**Ziel:** Die Daten verstehen, bevor der Prozess analysiert wird: Was steht in einer Zeile, welche Aktivitäten gibt es, und wie enden die Anträge?

**Vorgehen:** Für jede Frage wird zuerst eine Hypothese formuliert und dann mit den Daten geprüft.

## Daten laden

Das XES-Log wurde mit `src/load.py` einmalig eingelesen, geprüft und als Parquet gespeichert (lädt in ~1 s statt ~20 s).

In [1]:
from pathlib import Path

import pandas as pd

DATA_FILE = Path("..") / "data" / "processed" / "bpi2017.parquet"
df = pd.read_parquet(DATA_FILE)

## Erster Blick: Was steht in einer Zeile?

Jede Zeile ist ein **Event**: *welche* Aktivität (`concept:name`) *wer* (`org:resource`) *wann* (`time:timestamp`) in *welchem Antrag* (`case:concept:name`) ausgeführt hat.

In [2]:
df.head(10)

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID
0,Created,User_1,A_Create Application,Application,Application_652823628,complete,2016-01-01 09:51:15.304+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,None,NaN,None,NaN,NaN,NaN
1,statechange,User_1,A_Submitted,Application,ApplState_1582051990,complete,2016-01-01 09:51:15.352+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,None,NaN,None,NaN,NaN,NaN
2,Created,User_1,W_Handle leads,Workflow,Workitem_1298499574,schedule,2016-01-01 09:51:15.774+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,None,NaN,None,NaN,NaN,NaN
3,Deleted,User_1,W_Handle leads,Workflow,Workitem_1673366067,withdraw,2016-01-01 09:52:36.392+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,None,NaN,None,NaN,NaN,NaN
4,Created,User_1,W_Complete application,Workflow,Workitem_1493664571,schedule,2016-01-01 09:52:36.403+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,None,NaN,None,NaN,NaN,NaN
5,statechange,User_1,A_Concept,Application,ApplState_642383566,complete,2016-01-01 09:52:36.413+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,None,NaN,None,NaN,NaN,NaN
6,Obtained,User_17,W_Complete application,Workflow,Workitem_1875340971,start,2016-01-02 10:45:22.429+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,None,NaN,None,NaN,NaN,NaN
7,Released,User_17,W_Complete application,Workflow,Workitem_1452291795,suspend,2016-01-02 10:49:28.816+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,None,NaN,None,NaN,NaN,NaN
8,statechange,User_52,A_Accepted,Application,ApplState_99568828,complete,2016-01-02 11:23:04.299+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,None,NaN,None,NaN,NaN,NaN
9,Created,User_52,O_Create Offer,Offer,Offer_148581083,complete,2016-01-02 11:29:03.994+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,20000.0,44.0,True,498.29,True,979.0,20000.0,NaN


**Beobachtung:** Alle 10 Zeilen gehören zum selben Antrag (`Application_652823628`). Die ersten Schritte bucht `User_1` innerhalb von 80 Sekunden, vermutlich das System. Danach liegt der Antrag rund 25 Stunden in der Warteschlange, bevor ein Mitarbeiter (`User_17`) ihn 4 Minuten lang bearbeitet. Schon hier deutet sich an: Die Durchlaufzeit besteht vor allem aus Wartezeit (wird in Notebook 03 quantifiziert).

`lifecycle:transition` beschreibt den Zustand eines Arbeitsschritts: `schedule` (angelegt), `start`, `suspend` (pausiert), `complete`, `withdraw` (zurückgezogen).

## Datentypen und fehlende Werte

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1202267 entries, 0 to 1202266
Data columns (total 19 columns):
 #   Column                 Non-Null Count    Dtype              
---  ------                 --------------    -----              
 0   Action                 1202267 non-null  str                
 1   org:resource           1202267 non-null  str                
 2   concept:name           1202267 non-null  str                
 3   EventOrigin            1202267 non-null  str                
 4   EventID                1202267 non-null  str                
 5   lifecycle:transition   1202267 non-null  str                
 6   time:timestamp         1202267 non-null  datetime64[us, UTC]
 7   case:LoanGoal          1202267 non-null  str                
 8   case:ApplicationType   1202267 non-null  str                
 9   case:concept:name      1202267 non-null  str                
 10  case:RequestedAmount   1202267 non-null  float64            
 11  FirstWithdrawalAmount  42995 non-nu

Die Angebotsspalten (`OfferedAmount`, `CreditScore`, `MonthlyCost` …) sind nur in 42.995 von 1.202.267 Zeilen gefüllt.

**Hypothese:** Diese Werte stehen genau bei den Events, die ein Angebot erstellen (`O_Create Offer`). Wenn das stimmt, darf in den gefüllten Zeilen keine andere Aktivität vorkommen.

In [4]:
offer_rows = df[df["OfferedAmount"].notna()]
offer_rows["concept:name"].value_counts()

concept:name
O_Create Offer    42995
Name: count, dtype: int64

In [5]:
n_missing = df["OfferedAmount"].isna().sum()
n_create_offer = (df["concept:name"] == "O_Create Offer").sum()
n_missing == len(df) - n_create_offer

np.True_

**Ergebnis: bestätigt.** Die Angebotsdaten sind **Event-Attribute** von `O_Create Offer`. Leere Werte bedeuten also „nicht zutreffend", nicht „fehlend". Im Gegensatz dazu stehen **Case-Attribute** (Präfix `case:`) in jeder Zeile eines Antrags.

Außerdem: 42.995 Angebote bei 31.509 Anträgen, ein Antrag kann also mehrere Angebote erhalten.

## Welche Aktivitäten gibt es?

Die Präfixe zeigen drei Perspektiven auf den Prozess: `A_` = Application (Antragsstatus), `O_` = Offer (Angebote), `W_` = Workflow (Arbeitsschritte der Mitarbeitenden).

In [6]:
activity_counts = df["concept:name"].value_counts()
print(f"Anzahl verschiedener Aktivitäten: {len(activity_counts)}")
activity_counts


Anzahl verschiedener Aktivitäten: 26


concept:name
W_Validate application        209496
W_Call after offers           191092
W_Call incomplete files       168529
W_Complete application        148900
W_Handle leads                 47264
O_Create Offer                 42995
O_Created                      42995
O_Sent (mail and online)       39707
A_Validating                   38816
A_Create Application           31509
A_Concept                      31509
A_Accepted                     31509
A_Complete                     31362
O_Returned                     23305
A_Incomplete                   23055
O_Cancelled                    20898
A_Submitted                    20423
O_Accepted                     17228
A_Pending                      17228
A_Cancelled                    10431
O_Refused                       4695
A_Denied                        3753
W_Assess potential fraud        3282
O_Sent (online only)            2026
W_Shortened completion           238
W_Personal Loan collection        22
Name: count, dtype: int64

## Wie endet ein Antrag?

Kandidaten für den Ausgang sind die `A_`-Statusaktivitäten. (`O_`-Aktivitäten beziehen sich auf einzelne Angebote, nicht auf den ganzen Antrag.)

**Frage:** Bedeutet `A_Accepted` „Kredit bewilligt"?

In [7]:
candidates = ["A_Accepted", "A_Pending", "A_Denied", "A_Cancelled"]
for activity in candidates:
    n = df.loc[df["concept:name"] == activity, "case:concept:name"].nunique()
    print(f"{activity:12} kommt in {n:>6,} Anträgen vor")
    

A_Accepted   kommt in 31,509 Anträgen vor
A_Pending    kommt in 17,228 Anträgen vor
A_Denied     kommt in  3,752 Anträgen vor
A_Cancelled  kommt in 10,431 Anträgen vor


**Ergebnis:** `A_Accepted` kommt in **allen** 31.509 Anträgen vor, also auch in abgelehnten. Es bedeutet „zur Bearbeitung angenommen", nicht „bewilligt". Den Ausgang markieren:

- `A_Pending`: bewilligt (Kunde hat ein Angebot angenommen; gleich häufig wie `O_Accepted`)
- `A_Denied`: von der Bank abgelehnt
- `A_Cancelled`: abgebrochen

Zusammen ergeben sie 31.411 Anträge. Die übrigen 98 liefen am Ende der Datenerfassung (01.02.2017) noch. Diese **offenen Fälle** müssen bei Durchlaufzeiten später herausgefiltert werden.

### Ausgang je Antrag

Aus der Event-Tabelle wird eine Tabelle mit **einer Zeile pro Antrag** gebildet.

In [8]:
OUTCOMES = {"A_Pending": "approved", "A_Denied": "denied", "A_Cancelled": "cancelled"}

outcome_events = df[df["concept:name"].isin(OUTCOMES)]
outcome = outcome_events.groupby("case:concept:name")["concept:name"].first().map(OUTCOMES)
outcome.head()


case:concept:name
Application_1000086665    cancelled
Application_1000158214     approved
Application_1000311556    cancelled
Application_1000334415       denied
Application_1000339879     approved
Name: concept:name, dtype: str

In [9]:
cases = df.groupby("case:concept:name")[
    ["case:LoanGoal", "case:ApplicationType", "case:RequestedAmount"]
].first()
cases["outcome"] = outcome
cases["outcome"] = cases["outcome"].fillna("open")
cases["outcome"].value_counts(normalize=True).round(3)


outcome
approved     0.547
cancelled    0.331
denied       0.119
open         0.003
Name: proportion, dtype: float64

## Warum wird jeder dritte Antrag abgebrochen?

33 % der Anträge werden abgebrochen, fast dreimal so viele, wie die Bank ablehnt.

**Hypothese:** Die Kunden sagen aktiv ab, z. B. weil ihnen das Angebot nicht passt. Dann müssten die Abbrüche zu sehr unterschiedlichen Zeitpunkten passieren und überwiegend von Mitarbeitenden gebucht werden.

**Test 1:** Wer bucht den Abbruch?

In [10]:
cancel_events = df[df["concept:name"] == "A_Cancelled"]
cancel_events["org:resource"].value_counts(normalize=True).head(5).round(3)


org:resource
User_1     0.762
User_5     0.013
User_3     0.012
User_2     0.010
User_49    0.009
Name: proportion, dtype: float64

**Test 2:** Wie viel Zeit liegt zwischen dem letzten versendeten Angebot und dem Abbruch?

In [11]:
cancelled_ids = cancel_events["case:concept:name"]
events_cancelled = df[df["case:concept:name"].isin(cancelled_ids)]

sent_events = events_cancelled[events_cancelled["concept:name"].str.startswith("O_Sent")]
last_sent = sent_events.groupby("case:concept:name")["time:timestamp"].max()
cancelled_at = cancel_events.groupby("case:concept:name")["time:timestamp"].first()

days_until_cancel = (cancelled_at - last_sent).dt.total_seconds() / 86400
days_until_cancel.describe().round(1)


count    10321.0
mean        27.0
std         10.4
min          0.0
25%         30.5
50%         30.7
75%         30.8
max        167.2
Name: time:timestamp, dtype: float64

**Test 3:** Hatten die Kunden überhaupt reagiert, also ein Angebot zurückgeschickt (`O_Returned`)?

In [12]:
returned_ids = df.loc[df["concept:name"] == "O_Returned", "case:concept:name"].unique()
share_returned = cancelled_ids.isin(returned_ids).mean()
print(f"Anteil abgebrochener Anträge mit zurückgeschicktem Angebot: {share_returned:.1%}")

Anteil abgebrochener Anträge mit zurückgeschicktem Angebot: 9.2%


## Befund: Abbrüche entstehen durch Funkstille, nicht durch Absagen

- **33 %** aller Kreditanträge werden abgebrochen (zum Vergleich: 12 % abgelehnt, 55 % bewilligt).
- **76 %** der Abbrüche bucht `User_1`, sehr wahrscheinlich das System.
- Zwischen letztem Angebot und Abbruch liegen im Median **30,7 Tage**; die mittleren 50 % liegen zwischen 30,5 und 30,8 Tagen. Eine so enge Verteilung passt nicht zu individuellem Kundenverhalten, sondern zu einer festen Frist.
- Nur bei rund **9 %** der abgebrochenen Anträge hatte der Kunde ein Angebot zurückgeschickt.

**Interpretation:** Die Hypothese „Kunden sagen aktiv ab" ist widerlegt. Die Daten deuten stark darauf hin, dass das System Anträge automatisch storniert, wenn der Kunde etwa 30 Tage nach dem Angebot nicht reagiert. *(Diese Regel ist aus den Daten erschlossen, nicht dokumentiert.)*

**Mögliche Maßnahme:** Kunden vor Ablauf der Frist gezielt erinnern. Ob und wann die bestehende Aktivität `W_Call after offers` stattfindet und wirkt, wird in Notebook 03 untersucht.